In [1]:
import torch

print("cuda available:", torch.cuda.is_available())
print("xpu available:", hasattr(torch, "xpu") and torch.xpu.is_available())
print("cuda device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("xpu device:", torch.xpu.get_device_name(0) if hasattr(torch, "xpu") and torch.xpu.is_available() else None)


cuda available: True
xpu available: False
cuda device: NVIDIA GeForce RTX 5080
xpu device: None


In [2]:
from dnallm import load_config, load_model_and_tokenizer, DNADataset, DNATrainer

W0917 10:35:03.702000 28104 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [3]:
# Load the config file
configs = load_config("./finetune_config.yaml")

In [4]:
# Load the model and tokenizer
model_name = "zhangtaolab/plant-dnabert-BPE"
# from Hugging Face
# model, tokenizer = load_model_and_tokenizer(model_name, task_config=configs['task'], source="huggingface")
# from ModelScope
model, tokenizer = load_model_and_tokenizer(model_name, task_config=configs['task'], source="modelscope")

10:35:07 - dnallm.utils.support - INFO - Model files are stored in C:\Users\forre\.cache\modelscope\hub\models\zhangtaolab\plant-dnabert-BPE


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at C:\Users\forre\.cache\modelscope\hub\models\zhangtaolab\plant-dnabert-BPE and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
# Load the datasets
data_name = "zhangtaolab/plant-multi-species-core-promoters"
# from Hugging Face
# datasets = DNADataset.from_huggingface(data_name, seq_col="sequence", label_col="label", tokenizer=tokenizer, max_length=512)
# from ModelScope
datasets = DNADataset.from_modelscope(data_name, seq_col="sequence", label_col="label", tokenizer=tokenizer, max_length=512)

# Encode the datasets
datasets.encode_sequences()

# sample datasets
sampled_datasets = datasets.sampling(0.05, overwrite=True)

In [6]:
# Initialize the trainer
trainer = DNATrainer(
    model=model,
    config=configs,
    datasets=sampled_datasets
)

In [7]:
# Start training
metrics = trainer.train()
print(metrics)

Step,Training Loss,Validation Loss


{'train_runtime': 146.9949, 'train_samples_per_second': 67.921, 'train_steps_per_second': 4.245, 'total_flos': 2626900776714240.0, 'train_loss': 0.5002034016144581, 'epoch': 3.0}


In [8]:
# Do prediction on the test set
trainer.infer()

PredictionOutput(predictions=array([[ 1.6436427 , -1.5315328 ],
       [-1.2628831 ,  1.3836706 ],
       [-1.2190946 ,  1.1393472 ],
       [-0.29723454,  0.28245202],
       [ 1.6197702 , -1.5164536 ],
       [-1.0976777 ,  1.2118076 ],
       [ 1.2008334 , -1.1583742 ],
       [ 0.20511456, -0.4014332 ],
       [-0.2574762 ,  0.28829253],
       [-0.9469568 ,  1.0159262 ],
       [-1.2926592 ,  1.369565  ],
       [-1.2494377 ,  1.356768  ],
       [ 0.9328076 , -0.9508714 ],
       [-0.8619718 ,  0.76516455],
       [ 1.712569  , -1.5491533 ],
       [ 0.8009951 , -1.0779247 ],
       [ 1.5787581 , -1.4958701 ],
       [ 1.6159931 , -1.5978874 ],
       [ 1.5299057 , -1.4917235 ],
       [-1.361909  ,  1.5262853 ],
       [-0.8210836 ,  0.9714535 ],
       [-0.3355252 ,  0.4019934 ],
       [-1.4388356 ,  1.5639977 ],
       [ 0.7990718 , -1.0062908 ],
       [-1.4274888 ,  1.4505374 ],
       [-0.420633  ,  0.3760395 ],
       [ 1.4302207 , -1.7193104 ],
       [ 1.2333289 , -1.46